# 1.3 Transformers, what can they do?

The examples from [section 1.3](https://huggingface.co/learn/llm-course/chapter1/3) of the course, run locally with Transformers 5. Where Transformers 5 changed something, the cell says what changed.

Every model downloads once to `~/.cache/huggingface` (about 12 GB for the whole notebook; Qwen3-4B and Whisper large are the big ones). Kernel: `.env`.

In [1]:
import torch, transformers

print('transformers', transformers.__version__, '| torch', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

transformers 5.22.2 | torch 2.15.0+cu132
GPU: NVIDIA GeForce RTX 5090


## Working with pipelines

`pipeline()` connects a model with its preprocessing and postprocessing: text in, readable answer out.

In [2]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")
classifier("I've been waiting for a HuggingFace course my whole life.")

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


[{'label': 'POSITIVE', 'score': 0.9598046541213989}]

In [3]:
classifier(
    ["I've been waiting for a HuggingFace course my whole life.", "I hate this so much!"]
)

[{'label': 'POSITIVE', 'score': 0.9598046541213989},
 {'label': 'NEGATIVE', 'score': 0.9994558691978455}]

## Zero-shot classification

You choose the labels; the model was never fine-tuned on them.

In [4]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification")
classifier(
    "This is a course about the Transformers library",
    candidate_labels=["education", "politics", "business"],
)

[transformers] No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


{'sequence': 'This is a course about the Transformers library',
 'labels': ['education', 'business', 'politics'],
 'scores': [0.8445951342582703, 0.11197715252637863, 0.04342779889702797]}

## Text generation

Generation involves randomness, so your text will differ.

In [5]:
from transformers import pipeline

generator = pipeline("text-generation")
generator("In this course, we will teach you how to")

[transformers] No model was supplied, defaulted to HuggingFaceTB/SmolLM3-3B and revision a07cc9a.
Using a pipeline without specifying a model name and revision in production is not recommended.


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'generated_text': "In this course, we will teach you how to create a simple web scraper using Python and the BeautifulSoup library. Web scraping is the process of extracting data from websites. BeautifulSoup is a Python library that allows you to parse HTML and XML documents.\n\nWe will cover the following topics:\n\n* How to install the necessary libraries\n* How to send a HTTP request to a website\n* How to parse the HTML response\n* How to extract specific data from the HTML response\n* How to handle exceptions and errors\n\nAfter completing this course, you will be able to create a simple web scraper that extracts data from a website and stores it in a file. You will also learn how to handle exceptions and errors, which is important for writing robust web scrapers.\n\nTo get started, you will need to install the necessary libraries. You can install BeautifulSoup and requests using pip:\n\npip install requests beautifulsoup4\n\nNow, let's get started with the course!\n\n**Course O

## Using any model from the Hub in a pipeline

[HuggingFaceTB/SmolLM2-360M](https://huggingface.co/HuggingFaceTB/SmolLM2-360M), as in the course text (the course's own notebook still uses `distilgpt2`).

In [6]:
from transformers import pipeline

generator = pipeline("text-generation", model="HuggingFaceTB/SmolLM2-360M")
generator(
    "In this course, we will teach you how to",
    max_length=30,
    num_return_sequences=2,
)

[{'generated_text': 'In this course, we will teach you how to use the information you have learned to create web applications that meet the needs of your customers.\n\n'},
 {'generated_text': 'In this course, we will teach you how to find your own personal mission statement. It is a statement of purpose that you can use to guide your'}]

## Mask filling

`<mask>` is this model's mask token; BERT models use `[MASK]`.

In [7]:
from transformers import pipeline

unmasker = pipeline("fill-mask")
unmasker("This course will teach you all about <mask> models.", top_k=2)

[transformers] No model was supplied, defaulted to distilbert/distilroberta-base and revision fb53ab8.
Using a pipeline without specifying a model name and revision in production is not recommended.


[transformers] RobertaForMaskedLM LOAD REPORT from: distilbert/distilroberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'score': 0.19619813561439514,
  'token': 30412,
  'token_str': ' mathematical',
  'sequence': 'This course will teach you all about mathematical models.'},
 {'score': 0.04052721709012985,
  'token': 38163,
  'token_str': ' computational',
  'sequence': 'This course will teach you all about computational models.'}]

## Named entity recognition

`aggregation_strategy="simple"` regroups the pieces of each entity.

In [8]:
from transformers import pipeline

ner = pipeline("ner", aggregation_strategy="simple")
ner("My name is Sylvain and I work at Hugging Face in Brooklyn.")

[transformers] No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'entity_group': 'PER',
  'score': np.float32(0.9981694),
  'word': 'Sylvain',
  'start': 11,
  'end': 18},
 {'entity_group': 'ORG',
  'score': np.float32(0.9796019),
  'word': 'Hugging Face',
  'start': 33,
  'end': 45},
 {'entity_group': 'LOC',
  'score': np.float32(0.9932106),
  'word': 'Brooklyn',
  'start': 49,
  'end': 57}]

## Question answering, summarization and translation: removed in Transformers 5

The course uses `pipeline("question-answering")`, `pipeline("summarization")` and `pipeline("translation", ...)`. Transformers 5 removed all three (and `text2text-generation`): the [migration guide](https://github.com/huggingface/transformers/blob/main/MIGRATION_GUIDE_V5.md#pipelines) says a modern chat model with the `text-generation` pipeline gives much better results. The course's code now raises an error:

In [9]:
from transformers import pipeline

for task in ["question-answering", "summarization", "translation"]:
    try:
        pipeline(task)
    except KeyError as err:
        print(err, end="\n\n")

'The task question-answering has been removed from transformers. Use the `text-generation` pipeline with a chat model instead. See the migration guide for more details: https://github.com/huggingface/transformers/blob/main/MIGRATION_GUIDE_V5.md#pipelines'

'The task summarization has been removed from transformers. Use the `text-generation` pipeline with a chat model instead. See the migration guide for more details: https://github.com/huggingface/transformers/blob/main/MIGRATION_GUIDE_V5.md#pipelines'

'The task translation has been removed from transformers. Use the `text-generation` pipeline with a chat model instead. See the migration guide for more details: https://github.com/huggingface/transformers/blob/main/MIGRATION_GUIDE_V5.md#pipelines'



## The current way: one chat model, prompted for each task

`Qwen/Qwen3-4B-Instruct-2507` is the model in the migration guide's example (about 8 GB). Any chat model works; a smaller one needs less memory.

In [10]:
from transformers import pipeline

chat = pipeline("text-generation", model="Qwen/Qwen3-4B-Instruct-2507")

def ask(prompt):
    """Send one user message and return the reply."""
    messages = [{"role": "user", "content": prompt}]
    return chat(messages, max_new_tokens=200)[0]["generated_text"][-1]["content"]

Question answering. The old pipeline *extracted* the answer from the context; a chat model *generates* it, so the prompt asks it to stay within the context.

In [11]:
ask(
    "Answer the question using only the context.\n\n"
    "Context: My name is Sylvain and I work at Hugging Face in Brooklyn\n"
    "Question: Where do I work?"
)

'You work at Hugging Face in Brooklyn.'

Summarization, with the course's text.

In [12]:
summarizer_text = """
America has changed dramatically during recent years. Not only has the number of
graduates in traditional engineering disciplines such as mechanical, civil,
electrical, chemical, and aeronautical engineering declined, but in most of
the premier American universities engineering curricula now concentrate on
and encourage largely the study of engineering science. As a result, there
are declining offerings in engineering subjects dealing with infrastructure,
the environment, and related issues, and greater concentration on high
technology subjects, largely supporting increasingly complex scientific
developments. While the latter is important, it should not be at the expense
of more traditional engineering.

Rapidly developing economies such as China and India, as well as other
industrial countries in Europe and Asia, continue to encourage and advance
the teaching of engineering. Both China and India, respectively, graduate
six and eight times as many traditional engineers as does the United States.
Other industrial countries at minimum maintain their output, while America
suffers an increasingly serious decline in the number of engineering graduates
and a lack of well-educated engineers.
"""

ask("Summarize the following text in two sentences:\n" + summarizer_text)

"America has seen a significant decline in traditional engineering graduates and a shift toward engineering science, leading to reduced focus on infrastructure and environmental engineering. In contrast, rapidly developing nations like China and India are producing far more traditional engineers, highlighting America's growing shortfall in engineering education and workforce."

Translation, French to English.

In [13]:
ask("Translate into English: Ce cours est produit par Hugging Face.")

'This course is produced by Hugging Face.'

## Image and audio pipelines

In [14]:
from transformers import pipeline

image_classifier = pipeline(
    task="image-classification", model="google/vit-base-patch16-224"
)
result = image_classifier(
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
)
print(result)

[transformers] `ViTImageProcessor` requires torchvision (not installed); falling back to `ViTImageProcessorPil` for backward compatibility. Install torchvision to use the default backend, or import `ViTImageProcessorPil` directly to silence this warning.


[{'label': 'lynx, catamount', 'score': 0.433557391166687}, {'label': 'cougar, puma, catamount, mountain lion, painter, panther, Felis concolor', 'score': 0.034798379987478256}, {'label': 'snow leopard, ounce, Panthera uncia', 'score': 0.032396040856838226}, {'label': 'Egyptian cat', 'score': 0.023950539529323578}, {'label': 'tiger cat', 'score': 0.02287772111594677}]


In [15]:
from transformers import pipeline

transcriber = pipeline(
    task="automatic-speech-recognition", model="openai/whisper-large-v3"
)
result = transcriber(
    "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/mlk.flac"
)
print(result)

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.


[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.


[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


{'text': ' I have a dream that one day this nation will rise up and live out the true meaning of its creed.'}
